In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from pymongo import MongoClient
import time

# --------------------------------------------------
# 1. METADATOS DEL PROCESO
# --------------------------------------------------
NOMBRE_GRUPO = "G2_Ecommerce"
CATEGORIA_ACTUAL = "Young Adult"   # Aporte del primer código
LIMITE_PAGINAS = 3

# --------------------------------------------------
# 2. CONFIGURACION DEL NAVEGADOR
# --------------------------------------------------
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --------------------------------------------------
# 3. MEMORIA TEMPORAL PARA GUARDAR LOS DATOS EXTRAIDOS
# --------------------------------------------------
datos_finales = []

try:
    # Abrir el sitio inicial
    driver.get("http://books.toscrape.com/catalogue/category/books/young-adult_21/index.html")

    # Recorrer varias páginas
    for nivel_pagina in range(LIMITE_PAGINAS):
        print(f"--- Procesando página {nivel_pagina + 1} de categoría {CATEGORIA_ACTUAL} ---")

        # Esperar a que carguen los libros
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )

        # Capturar todos los bloques de libros
        bloques_libros = driver.find_elements(By.CLASS_NAME, "product_pod")

        for bloque in bloques_libros:
            try:
                # Extraer título completo y precio
                titulo = bloque.find_element(By.TAG_NAME, "h3") \
                              .find_element(By.TAG_NAME, "a") \
                              .get_attribute("title")

                precio_texto = bloque.find_element(By.CLASS_NAME, "price_color").text

                # Crear estructura uniforme (integrando el aporte del primer código)
                item_extraido = {
                    "item": titulo,
                    "valor": precio_texto,  # todavía en texto, luego se limpia
                    "categoria": CATEGORIA_ACTUAL,
                    "pagina": nivel_pagina + 1,
                    "grupo": NOMBRE_GRUPO,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
                }

                datos_finales.append(item_extraido)

            except Exception as e:
                print("Error extrayendo un libro:", e)

        # Intentar pasar a la siguiente página
        try:
            boton_siguiente = driver.find_element(By.CSS_SELECTOR, "li.next a")
            boton_siguiente.click()
        except Exception:
            print("No hay más páginas disponibles.")
            break

    print(f"\nExtracción finalizada. Registros capturados en memoria: {len(datos_finales)}")

except Exception as e:
    print("Fallo en el scraping:", e)

finally:
    driver.quit()

# --------------------------------------------------
# 4. CONEXION A MONGODB
# --------------------------------------------------
try:
    client = MongoClient("database", 27017)
    db = client["proyecto_bigdata"]
    coleccion = db["datos_libros"]
    print("CONEXION EXITOSA: Conectado a MongoDB.")
except Exception as e:
    print("ERROR DE CONEXION:", e)

# --------------------------------------------------
# 5. LIMPIEZA E INSERCION DE DATOS
# --------------------------------------------------
registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        # Limpiar el precio y convertirlo a float
        valor_sucio = str(dato.get("valor", "0"))
        valor_limpio = valor_sucio.replace("£", "").replace(",", "").strip()
        dato["valor"] = float(valor_limpio)

        # Insertar en MongoDB
        coleccion.insert_one(dato)
        registros_exitosos += 1

    except ValueError:
        print("ADVERTENCIA: No se pudo convertir el valor:", dato.get("valor"))
        registros_fallidos += 1

    except Exception as e:
        print("ERROR EN INSERCION:", e)
        registros_fallidos += 1

# --------------------------------------------------
# 6. RESUMEN FINAL
# --------------------------------------------------
print("\nRESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)
print("PROCESO FINALIZADO: Datos enviados a MongoDB.")

--- Procesando página 1 de categoría Young Adult ---
--- Procesando página 2 de categoría Young Adult ---
--- Procesando página 3 de categoría Young Adult ---
No hay más páginas disponibles.

Extracción finalizada. Registros capturados en memoria: 54
CONEXION EXITOSA: Conectado a MongoDB.
ERROR EN INSERCION: database:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69e58206dea775ce820d5c50, topology_type: Unknown, servers: [<ServerDescription ('database', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('database:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>
ERROR EN INSERCION: database:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69e5